# Notebook de Mise à Jour des Valeurs Réelles

Ce notebook permet de mettre à jour les prédictions de consommation électrique avec les valeurs réelles de la veille.

## Fonctionnalités :
1. Configuration de la base de données
2. Récupération des prédictions de la veille
3. Génération de valeurs aléatoires (pour l'instant)
4. Mise à jour des enregistrements dans la base de données
5. Vérification des mises à jour

## Structure de la table de stockage :
- `prediction_id`: TEXT (UUID)
- `prediction_timestamp`: TIMESTAMP
- `prediction_date`: TIMESTAMP
- `prediction_index`: INTEGER
- `prediction`: DOUBLE PRECISION
- `confidence`: DOUBLE PRECISION
- `model_version`: TEXT
- `actual_value`: DOUBLE PRECISION (nouveau champ)
- `created_at`: TIMESTAMP

## 0. Initialisation et Configuration

In [1]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


## 1. Configuration de la Base de Données

In [2]:
# Configuration PostgreSQL (à adapter selon votre environnement)
DB_URI = os.getenv("PREDICTIONS_POSTGRES_URI")

if DB_URI:
    print("📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI")
else:
    DB_CONFIG = {
        "host": os.getenv("DB_HOST", "localhost"),
        "port": os.getenv("DB_PORT", "5432"),
        "database": os.getenv("DB_NAME", "jinsudai"),
        "user": os.getenv("DB_USER", "postgres"),
        "password": os.getenv("DB_PASSWORD", ""),
    }
    print(f"📊 Configuration DB :")
    print(f"  Host: {DB_CONFIG['host']}")
    print(f"  Port: {DB_CONFIG['port']}")
    print(f"  Database: {DB_CONFIG['database']}")
    
    # Construire l'URI PostgreSQL
    DB_URI = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    print(f"  URI construite")

📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI


## 2. Initialisation du Pipeline de Valeurs Réelles

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd() / "src"))

from ml.pipelines.ingestion_pipeline import IngestionPipeline

# Initialiser le pipeline
pipeline = IngestionPipeline(db_uri=DB_URI)

print("🔧 Pipeline d'ingestion des valeurs réelles initialisé")
print(f"  URI DB: {'Configurée' if DB_URI else 'Non configurée'}")

c:\Users\SustCoop\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\ml-6PV5fMfH-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-14 01:07:54,005 - ml.utils.pipelines.Actual_values_pipeline - INFO - Pipeline de valeurs réelles initialisé


🔧 Pipeline de valeurs réelles initialisé
  URI DB: Configurée


## 3. Configuration du Pipeline

In [4]:
print("🔧 Configuration du pipeline...")

try:
    if not pipeline.setup():
        raise RuntimeError("Échec de la configuration du pipeline")
    
    print("✅ Pipeline configuré avec succès")
    print("  - Connexion à la base de données vérifiée")
    print("  - Colonne actual_value ajoutée si nécessaire")
    
except Exception as e:
    logger.error(f"Erreur lors de la configuration: {e}")
    raise

2026-06-14 01:07:54,048 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 1: CONFIGURATION ===


🔧 Configuration du pipeline...


2026-06-14 01:07:54,401 - ml.utils.pipelines.database_handler - INFO - Connexion à la base de données vérifiée
2026-06-14 01:07:54,631 - ml.utils.pipelines.database_handler - INFO - Colonne actual_value ajoutée ou déjà existante
2026-06-14 01:07:54,634 - ml.utils.pipelines.Actual_values_pipeline - INFO - Base de données configurée


✅ Pipeline configuré avec succès
  - Connexion à la base de données vérifiée
  - Colonne actual_value ajoutée si nécessaire


## 4. Récupération des Prédictions de la Veille

In [5]:
from datetime import datetime, timedelta

# Calculer la date de la veille
today = datetime.now().date()
yesterday = today - timedelta(days=1)

print(f"📅 Récupération des prédictions du {yesterday}")
print(f"  Date actuelle: {today}")

try:
    if not pipeline.get_previous_day_predictions():
        print(f"⚠️ Aucune prédiction trouvée pour la date {yesterday}")
        print("   Assurez-vous que le pipeline de prédiction a été exécuté pour cette date.")
    else:
        print(f"✅ {len(pipeline.previous_day_predictions)} prédictions récupérées")
        
        # Afficher un aperçu des prédictions
        import pandas as pd
        display(pipeline.previous_day_predictions.head())
        
except Exception as e:
    logger.error(f"Erreur lors de la récupération des prédictions: {e}")
    raise

2026-06-14 01:07:54,675 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 2: RÉCUPÉRATION DES PRÉDICTIONS DE LA VEILLE ===
2026-06-14 01:07:54,677 - ml.utils.pipelines.Actual_values_pipeline - INFO - Récupération des prédictions du 2026-06-13
2026-06-14 01:07:54,681 - ml.utils.pipelines.Actual_values_pipeline - INFO - Plage de temps: 2026-06-13 00:00:00 à 2026-06-13 23:59:59.999999


📅 Récupération des prédictions du 2026-06-13
  Date actuelle: 2026-06-14


2026-06-14 01:07:54,898 - ml.utils.pipelines.Actual_values_pipeline - INFO - 72 prédictions récupérées pour la veille


✅ 72 prédictions récupérées


,prediction_id,prediction_timestamp,prediction_date,prediction_index,prediction,confidence,model_version,entity_id,run_id,actual_value
0,8cbf8689-98f4-4a12-9e73-3ee9664cdad3,2026-06-13 23:00:00,2026-06-13 23:00:00,48,604.445312,None,13,550e8400-e29b-41d4-a716-446655440000,e5c4f5a023914c958cdc3829bcf64e15,1250.0
1,a2edfb41-9ef0-4d58-822a-c47de8d09ca9,2026-06-13 23:00:00,2026-06-13 23:00:00,48,604.445312,None,13,550e8400-e29b-41d4-a716-446655440000,e5c4f5a023914c958cdc3829bcf64e15,1250.0
2,8f2136fc-1355-4530-9dcb-fe72d0364853,2026-06-13 23:00:00,2026-06-13 23:00:00,48,604.445312,None,13,550e8400-e29b-41d4-a716-446655440000,e5c4f5a023914c958cdc3829bcf64e15,1250.0
3,ea755ae5-208b-46bd-b58e-bcd86cbe92d5,2026-06-13 22:00:00,2026-06-13 22:00:00,47,727.927429,None,13,550e8400-e29b-41d4-a716-446655440000,e5c4f5a023914c958cdc3829bcf64e15,490.0
4,103be5db-07a4-41e9-9607-199e0aaacfe0,2026-06-13 22:00:00,2026-06-13 22:00:00,47,727.927429,None,13,550e8400-e29b-41d4-a716-446655440000,e5c4f5a023914c958cdc3829bcf64e15,490.0


## 5. Génération de Valeurs Aléatoires

In [6]:
print("🎲 Génération de valeurs aléatoires...")
print("  (variation de ±20% autour de la valeur prédite)")

try:
    if pipeline.previous_day_predictions is None or len(pipeline.previous_day_predictions) == 0:
        print("⚠️ Aucune prédiction à traiter")
    else:
        if not pipeline.generate_random_actual_values():
            raise RuntimeError("Échec de la génération des valeurs aléatoires")
        
        print(f"✅ {pipeline.updated_count} valeurs aléatoires générées et mises à jour")
        
except Exception as e:
    logger.error(f"Erreur lors de la génération des valeurs: {e}")
    raise

2026-06-14 01:07:55,148 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 3: GÉNÉRATION DES VALEURS ALÉATOIRES ===
2026-06-14 01:07:55,169 - ml.utils.pipelines.Actual_values_pipeline - INFO - 72 valeurs aléatoires générées


🎲 Génération de valeurs aléatoires...
  (variation de ±20% autour de la valeur prédite)


2026-06-14 01:07:55,751 - ml.utils.pipelines.database_handler - INFO - 72 prédictions mises à jour avec les valeurs réelles


✅ 72 valeurs aléatoires générées et mises à jour


## 6. Vérification des Mises à Jour

In [7]:
print("🔍 Vérification des mises à jour...")

try:
    updated_predictions = pipeline.verify_updates()
    
    if updated_predictions is None:
        print("⚠️ Impossible de vérifier les mises à jour")
    else:
        # Filtrer les enregistrements avec des valeurs réelles
        with_actual_values = updated_predictions[updated_predictions['actual_value'].notna()]
        
        print(f"✅ Vérification terminée")
        print(f"  Total prédictions: {len(updated_predictions)}")
        print(f"  Avec valeurs réelles: {len(with_actual_values)}")
        
        if len(with_actual_values) > 0:
            print("\n📊 Aperçu des prédictions mises à jour:")
            display(with_actual_values[['prediction_timestamp', 'prediction', 'actual_value']].head(10))
            
            # Calculer les statistiques
            print("\n📈 Statistiques de comparaison:")
            print(f"  Moyenne prédite: {with_actual_values['prediction'].mean():.2f}")
            print(f"  Moyenne réelle: {with_actual_values['actual_value'].mean():.2f}")
            print(f"  Écart moyen: {abs(with_actual_values['prediction'] - with_actual_values['actual_value']).mean():.2f}")
            print(f"  Écart relatif moyen: {(abs(with_actual_values['prediction'] - with_actual_values['actual_value']) / with_actual_values['prediction'] * 100).mean():.2f}%")
        
except Exception as e:
    logger.error(f"Erreur lors de la vérification: {e}")
    raise

2026-06-14 01:07:55,818 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 4: VÉRIFICATION DES MISES À JOUR ===


🔍 Vérification des mises à jour...


2026-06-14 01:07:56,317 - ml.utils.pipelines.Actual_values_pipeline - INFO - 72 prédictions avec des valeurs réelles
2026-06-14 01:07:56,331 - ml.utils.pipelines.Actual_values_pipeline - INFO - Exemple de valeurs mises à jour:
  prediction_timestamp  prediction  actual_value
0  2026-06-13 23:00:00  604.445312    577.785971
1  2026-06-13 23:00:00  604.445312    515.144805
2  2026-06-13 23:00:00  604.445312    606.027374
3  2026-06-13 22:00:00  727.927429    687.047334
4  2026-06-13 22:00:00  727.927429    596.992310


✅ Vérification terminée
  Total prédictions: 72
  Avec valeurs réelles: 72

📊 Aperçu des prédictions mises à jour:


,prediction_timestamp,prediction,actual_value
0,2026-06-13 23:00:00,604.445312,577.785971
1,2026-06-13 23:00:00,604.445312,515.144805
2,2026-06-13 23:00:00,604.445312,606.027374
3,2026-06-13 22:00:00,727.927429,687.047334
4,2026-06-13 22:00:00,727.927429,596.992310
5,2026-06-13 22:00:00,727.927429,856.621078
6,2026-06-13 21:00:00,723.122314,828.452848
7,2026-06-13 21:00:00,723.122314,861.523886
8,2026-06-13 21:00:00,723.122314,861.889774
9,2026-06-13 20:00:00,648.645081,596.851770



📈 Statistiques de comparaison:
  Moyenne prédite: 663.86
  Moyenne réelle: 668.58
  Écart moyen: 74.67
  Écart relatif moyen: 11.10%


## 7. Exécution Complète du Pipeline

In [8]:
print("🚀 Exécution complète du pipeline de valeurs réelles...")

try:
    success, results = pipeline.run_full_pipeline()
    
    if success:
        print("\n✅ Pipeline exécuté avec succès")
        print(f"  Enregistrements mis à jour: {pipeline.updated_count}")
    else:
        print("\n❌ Pipeline échoué")
        
except Exception as e:
    logger.error(f"Erreur lors de l'exécution du pipeline: {e}")
    raise

2026-06-14 01:07:56,453 - ml.utils.pipelines.Actual_values_pipeline - INFO - ####################################################
2026-06-14 01:07:56,455 - ml.utils.pipelines.Actual_values_pipeline - INFO - ### PIPELINE DE MISE À JOUR DES VALEURS RÉELLES ###
2026-06-14 01:07:56,461 - ml.utils.pipelines.Actual_values_pipeline - INFO - ### Date/Heure: 2026-06-14 01:07:56 ###
2026-06-14 01:07:56,463 - ml.utils.pipelines.Actual_values_pipeline - INFO - ####################################################

2026-06-14 01:07:56,465 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 1: CONFIGURATION ===


🚀 Exécution complète du pipeline de valeurs réelles...


2026-06-14 01:07:56,800 - ml.utils.pipelines.database_handler - INFO - Connexion à la base de données vérifiée
2026-06-14 01:07:57,151 - ml.utils.pipelines.database_handler - INFO - Colonne actual_value ajoutée ou déjà existante
2026-06-14 01:07:57,154 - ml.utils.pipelines.Actual_values_pipeline - INFO - Base de données configurée
2026-06-14 01:07:57,156 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 2: RÉCUPÉRATION DES PRÉDICTIONS DE LA VEILLE ===
2026-06-14 01:07:57,159 - ml.utils.pipelines.Actual_values_pipeline - INFO - Récupération des prédictions du 2026-06-13
2026-06-14 01:07:57,161 - ml.utils.pipelines.Actual_values_pipeline - INFO - Plage de temps: 2026-06-13 00:00:00 à 2026-06-13 23:59:59.999999
2026-06-14 01:07:57,423 - ml.utils.pipelines.Actual_values_pipeline - INFO - 72 prédictions récupérées pour la veille
2026-06-14 01:07:57,426 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 3: GÉNÉRATION DES VALEURS ALÉATOIRES ===
2026-06-14 01:07:57,445


✅ Pipeline exécuté avec succès
  Enregistrements mis à jour: 72


## 8. Nettoyage et Statistiques Finales

In [ ]:
# Statistiques finales de la base de données
from ml.pipelines.database_handler import DatabaseHandler

db_handler = DatabaseHandler(DB_URI)
stats = db_handler.get_prediction_stats()

if stats:
    print("📊 Statistiques finales de la base de données:")
    print(f"  Total des prédictions: {stats['total_predictions']}")
    print(f"  Table existe: {stats['table_exists']}")
else:
    print("⚠️ Impossible de récupérer les statistiques")

📊 Statistiques finales de la base de données:
  Total des prédictions: 264
  Table existe: True


## Résumé

Ce notebook a démontré :
1. ✅ Configuration du pipeline de valeurs réelles
2. ✅ Récupération des prédictions de la veille
3. ✅ Génération de valeurs aléatoires (±20% de variation)
4. ✅ Mise à jour des enregistrements dans la base de données
5. ✅ Vérification des mises à jour